In [21]:
# --- Model 1: Feed-Forward Neural Network on flattened spectrograms ---
# Inputs ready from Section 3.5:
#   Xf_train (6393, 16384) | Xf_val | Xf_test   flattened pixels
#   y_train / y_val / y_test                    integer genre_id
#
# TBD

# TODO: plot train vs val loss/accuracy

# When fitted, score it through the shared table:
# record("Feed-Forward NN", KerasModel, Xf_train, Xf_val, "2 hidden layers")
# plot_confusion(y_val, _labels(model, Xc_val), "<model name> - validation")

## Model 1: Feed-Forward Neural Network on Flattened Spectrogram

In [22]:
# load modules needed

import numpy as np
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from keras import metrics
tf.get_logger().setLevel('INFO')

---
### Step 1: Data Ingesetion
#### Use flattened normalized data from Step 3.5

Prior to this step, data has been split into training, validation, and test sets, and has been pre-processed to be ready for model training


In [23]:
# load in from local data
d = np.load("../data/local/splits.npz", allow_pickle=True)

Xf_train, Xf_val, Xf_test = d["Xf_train"], d["Xf_val"], d["Xf_test"]
y_train, y_val, y_test = d["y_train"], d["y_val"], d["y_test"]


In [24]:
# Xf_* are (n, 16384) = (n, 128 mel, 128 time) flattened
def to3d(X): return X.reshape(len(X), 128, 128)

tr = to3d(Xf_train)
mu = tr.mean(axis=(0, 2), keepdims=True)         # one mean per mel band
sd = tr.std(axis=(0, 2), keepdims=True) + 1e-6

def std_flat(X): return ((to3d(X) - mu) / sd).reshape(len(X), -1)

Xs_train, Xs_val, Xs_test = std_flat(Xf_train), std_flat(Xf_val), std_flat(Xf_test)

---
### Step 2: Build Model
#### Builds a feed-forward neural network that can have vary in many ways allowing for experimental testing

In [25]:
def build_model(n_classes,num_features,
                hidden_layer_sizes=[],
                activation='relu',
                optimizer='Adam',
                learning_rate=1e-4,
                metric='metric'
                ):
    """
    Args:
    n_classes: Number of output classes in the dataset.
    n_features: Number of features in the input dataset.
    hidden_layer_sizes: A list with the number of units in each hidden layer.
    activation: The activation function to use for the hidden layers.
    optimizer: The optimizer to use (SGD, Adam).
    learning_rate: The desired learning rate for the optimizer.
    metric: The desired metric.

    Returns:
    model: A tf.keras model (graph).
    """
    tf.keras.backend.clear_session()
    np.random.seed(0)
    tf.random.set_seed(0)

    # YOUR CODE HERE

    if optimizer == 'SGD':
        opt = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer == 'Adam':
        opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model = tf.keras.Sequential()



    # initial layer
    model.add(tf.keras.Input(shape=(num_features,), name='Input'))


    # loop through the hidden layers
    for l in range(len(hidden_layer_sizes)):
        layer_output = f'Output{l}'
        model.add(tf.keras.layers.Dense(units=hidden_layer_sizes[l], activation=activation, name=layer_output))
        model.add(tf.keras.layers.Dropout(0.5))



        # create output layer
    model.add(tf.keras.layers.Dense(
            units=n_classes,  
            use_bias=True,
            activation=None, 
            name='Output'
            ))


    model.compile(
        optimizer = opt,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  
        metrics = [metric]
    )

    return model

---
### Step 3: Experiment with Hyperparameters
#### Build function to train the model on input hyperparameters, and output the accuracy of the train and validation data sets

In [ ]:
def train_and_evaluate(hidden_layer_sizes=[],
                       activation='relu',
                       optimizer='Adam',
                       learning_rate=1e-4,
                       metric='accuracy',
                       num_epochs=100,
                       plot_on = 1):


    # Build the model.
    model = build_model(n_classes=len(np.unique(y_train)),num_features = Xs_train.shape[1],
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation=activation,
                        optimizer=optimizer,
                        metric=metric,
                        learning_rate=learning_rate)
    return model

# run the function
model = train_and_evaluate(hidden_layer_sizes = [128,64])

In [43]:
   early = tf.keras.callbacks.EarlyStopping(
       monitor="val_loss", patience=8, restore_best_weights=True)

   history = model.fit(Xs_train, y_train, epochs=100, batch_size=32,
                       validation_data=(Xs_val, y_val),
                       callbacks=[early], verbose=0)

   best = int(np.argmin(history.history['val_loss']))
   train_accuracy = history.history['accuracy']
   val_accuracy = history.history['val_accuracy']
   print('Best epoch %d | train %.4f | val %.4f'
         % (best + 1, train_accuracy[best], val_accuracy[best]))

Best epoch 7 | train 0.1774 | val 0.2950


---
### Step 4: Train a second model that averages over the nearest 2x2 grid


In [45]:
# --- 2x2 average pooling: (n, 16384) -> (n, 4096) ---
def pool2(X):
    # (n, 128 mel, 128 time) -> (n, 64, 2, 64, 2) -> average each 2x2 block
    X3 = X.reshape(len(X), 64, 2, 64, 2).mean(axis=(2, 4))   # (n, 64, 64)
    return X3.reshape(len(X), -1)                             # (n, 4096)

Xp_train, Xp_val, Xp_test = pool2(Xs_train), pool2(Xs_val), pool2(Xs_test)
print(Xp_train.shape, Xp_val.shape, Xp_test.shape)   # expect (6393, 4096) (800, 4096) (800, 4096)

(6393, 4096) (800, 4096) (800, 4096)


In [ ]:
# --- Fresh model, same setup as before but on the pooled input ---
model_p = build_model(
    n_classes=len(np.unique(y_train)),
    num_features=Xp_train.shape[1],
    hidden_layer_sizes=[256, 128],
    activation='relu',
    optimizer='Adam',
    learning_rate=1e-4,
    metric='accuracy',
)

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True)

history_p = model_p.fit(Xp_train, y_train, epochs=100, batch_size=32,
                        validation_data=(Xp_val, y_val),
                        callbacks=[early], verbose=0)

best = int(np.argmin(history_p.history['val_loss']))
print('Best epoch %d of %d | val acc %.4f'
      % (best + 1, len(history_p.history['val_loss']),
         history_p.history['val_accuracy'][best]))

# Fair comparison: both evaluated with dropout off
print("train:", model_p.evaluate(Xp_train, y_train, verbose=0)[1])
print("val:  ", model_p.evaluate(Xp_val,   y_val,   verbose=0)[1])


Best epoch 6 of 14 | val acc 0.3450
train: 0.44642576575279236
val:   0.3449999988079071
